# Chapter 2: Systems Foundations — Hardware for Running Agents

> **Book:** Agentic AI Engineering  
> **Part:** 1 — Foundations  
> **System:** Guardian Angel System (GAS) — Multi-agent diabetic patient supervision

In [ ]:
# Install dependencies
# pip install torch transformers matplotlib numpy psutil

In [ ]:
import os
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"
print(f"TEST_MODE: {TEST_MODE}")
if TEST_MODE:
    print("Running in TEST_MODE — GPU benchmarks will use simulated data.")

---
## 🎯 1. Chapter Goal

By the end of this notebook you will:

- Calculate **GPU memory requirements** for any LLM (weights + KV cache + activations)
- Understand **batching strategies** and their throughput/latency tradeoffs
- Benchmark **tokens/sec** for different batch sizes
- Build a **cost estimator** for production LLM inference
- Plan the **GAS deployment infrastructure** for 5 concurrent agents

**Why it matters:** Running 5 GAS agents simultaneously requires careful resource planning. An A100 (80GB) can run multiple 7B models, but you need to know the math.

---
## 📖 2. Theory Recap

### 2.1 GPU Memory Hierarchy

```
GPU Memory (VRAM) — Fast, limited (24-80 GB)
├── Model Weights:     params × bytes_per_param
├── KV Cache:          2 × layers × heads × head_dim × seq_len × batch × bytes
├── Activations:       batch × seq_len × hidden_dim × bytes (temporary)
└── Optimizer States:  2-3× model size (training only)

CPU RAM — Slower, large (256+ GB)
└── Offloaded layers, dataset, preprocessing
```

### 2.2 KV Cache Memory Math

For a model with:
- `L` layers, `H` attention heads, `d` head dimension
- Sequence length `S`, batch size `B`, precision `P` bytes

$$\text{KV Cache} = 2 \times L \times H \times d \times S \times B \times P$$

The factor of 2 is for Keys AND Values.

**Example (Llama-7B, fp16, seq=2048, batch=1):**
- L=32, H=32, d=128, S=2048, B=1, P=2 bytes
- KV Cache = 2 × 32 × 32 × 128 × 2048 × 1 × 2 = **1.07 GB**

### 2.3 Batching Strategies

| Strategy | Description | Pros | Cons |
|----------|-------------|------|------|
| **Static batching** | Fixed batch size, wait for full batch | Simple | High latency, GPU waste |
| **Dynamic batching** | Batch requests within time window | Better utilization | Complex |
| **Continuous batching** | Add/remove sequences mid-batch | Best throughput | Requires special runtime |
| **PagedAttention** | Virtual memory for KV cache | No fragmentation | vLLM-specific |

### 2.4 Throughput vs Latency Tradeoff

- **Throughput** (tokens/sec): maximize by using large batches
- **Latency** (ms/request): minimize by using small batches
- **Sweet spot**: continuous batching finds the Pareto frontier

### 2.5 Precision Formats

| Format | Bytes | Memory | Quality |
|--------|-------|--------|--------|
| fp32 | 4 | 100% | Best |
| fp16/bf16 | 2 | 50% | Near-identical |
| int8 | 1 | 25% | Slight degradation |
| int4 | 0.5 | 12.5% | Noticeable degradation |

---
## 🔨 3. Build It

### 3.1 GPU Memory Calculator

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

@dataclass
class ModelConfig:
    """Configuration for a transformer model."""
    name: str
    params_billions: float
    n_layers: int
    n_heads: int
    head_dim: int
    hidden_dim: int


def estimate_model_memory(
    params_billions: float,
    precision: str = 'fp16',
    seq_len: int = 2048,
    batch_size: int = 1,
    n_layers: Optional[int] = None,
    n_heads: Optional[int] = None,
    head_dim: Optional[int] = None,
    include_optimizer: bool = False
) -> dict:
    """
    Estimate GPU memory requirements for an LLM.
    
    Args:
        params_billions: Model size in billions of parameters
        precision: 'fp32', 'fp16', 'bf16', 'int8', or 'int4'
        seq_len: Maximum sequence length
        batch_size: Inference batch size
        n_layers: Number of transformer layers (estimated if None)
        n_heads: Number of attention heads (estimated if None)
        head_dim: Head dimension (estimated if None)
        include_optimizer: Include optimizer states (for training)
    
    Returns:
        dict with memory breakdown in GB
    """
    # Bytes per parameter
    precision_bytes = {'fp32': 4, 'fp16': 2, 'bf16': 2, 'int8': 1, 'int4': 0.5}
    bytes_per_param = precision_bytes.get(precision, 2)
    
    # Model weights
    params = params_billions * 1e9
    weights_gb = (params * bytes_per_param) / (1024**3)
    
    # Estimate architecture if not provided (rough heuristics)
    if n_layers is None:
        n_layers = int(4 * params_billions + 16)  # rough estimate
        n_layers = min(n_layers, 80)  # cap at 80
    if n_heads is None:
        n_heads = min(int(params_billions * 4 + 8), 64)
    if head_dim is None:
        head_dim = 128  # standard
    
    # KV Cache: 2 × layers × heads × head_dim × seq_len × batch × bytes
    kv_cache_bytes = 2 * n_layers * n_heads * head_dim * seq_len * batch_size * bytes_per_param
    kv_cache_gb = kv_cache_bytes / (1024**3)
    
    # Activations (rough estimate: ~1% of model size per batch)
    activations_gb = weights_gb * 0.01 * batch_size
    
    # Optimizer states (Adam: 2× model size for momentum + variance)
    optimizer_gb = weights_gb * 2 if include_optimizer else 0
    
    total_gb = weights_gb + kv_cache_gb + activations_gb + optimizer_gb
    
    # GPU recommendations
    gpus = {
        'RTX 3090 (24GB)': 24,
        'RTX 4090 (24GB)': 24,
        'A100 (40GB)': 40,
        'A100 (80GB)': 80,
        'H100 (80GB)': 80,
        'H100 (94GB SXM)': 94,
    }
    
    fits_on = [name for name, vram in gpus.items() if vram >= total_gb * 1.1]  # 10% headroom
    
    return {
        'model_size': f'{params_billions}B',
        'precision': precision,
        'weights_gb': round(weights_gb, 2),
        'kv_cache_gb': round(kv_cache_gb, 2),
        'activations_gb': round(activations_gb, 3),
        'optimizer_gb': round(optimizer_gb, 2),
        'total_gb': round(total_gb, 2),
        'seq_len': seq_len,
        'batch_size': batch_size,
        'fits_on_single_gpu': fits_on,
        'n_layers_used': n_layers,
        'n_heads_used': n_heads,
    }


# Test with common models
models = [
    (0.117, 'distilgpt2'),
    (1.3, 'GPT-2 XL / Phi-1.5'),
    (7.0, 'Llama-2-7B / Mistral-7B'),
    (13.0, 'Llama-2-13B'),
    (70.0, 'Llama-2-70B'),
]

print("GPU Memory Requirements (fp16, seq_len=2048, batch=1)")
print("=" * 75)
print(f"{'Model':<30} {'Weights':>10} {'KV Cache':>10} {'Total':>10} {'Fits on'}")
print("-" * 75)

for params, name in models:
    result = estimate_model_memory(params, precision='fp16', seq_len=2048)
    fits = result['fits_on_single_gpu'][0] if result['fits_on_single_gpu'] else 'Multi-GPU'
    print(f"{name:<30} {result['weights_gb']:>8.1f}GB {result['kv_cache_gb']:>8.2f}GB {result['total_gb']:>8.1f}GB  {fits}")

In [ ]:
# Visualize memory breakdown for 7B model across precisions
precisions = ['fp32', 'fp16', 'int8', 'int4']
model_size = 7.0

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Memory breakdown by precision
ax1 = axes[0]
weights_list, kv_list, act_list = [], [], []
for prec in precisions:
    r = estimate_model_memory(model_size, precision=prec)
    weights_list.append(r['weights_gb'])
    kv_list.append(r['kv_cache_gb'])
    act_list.append(r['activations_gb'])

x = np.arange(len(precisions))
width = 0.6
p1 = ax1.bar(x, weights_list, width, label='Weights', color='#3498db')
p2 = ax1.bar(x, kv_list, width, bottom=weights_list, label='KV Cache', color='#e74c3c')
p3 = ax1.bar(x, act_list, width, 
             bottom=[w+k for w,k in zip(weights_list, kv_list)], 
             label='Activations', color='#2ecc71')

ax1.axhline(y=24, color='orange', linestyle='--', label='RTX 4090 (24GB)')
ax1.axhline(y=80, color='red', linestyle='--', label='A100/H100 (80GB)')
ax1.set_xticks(x)
ax1.set_xticklabels(precisions)
ax1.set_ylabel('Memory (GB)')
ax1.set_title(f'7B Model Memory by Precision\n(seq_len=2048, batch=1)', fontweight='bold')
ax1.legend(loc='upper right')

# KV cache scaling with sequence length
ax2 = axes[1]
seq_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768]
for prec, color in [('fp16', '#3498db'), ('int8', '#e74c3c'), ('int4', '#2ecc71')]:
    kv_sizes = [estimate_model_memory(7.0, precision=prec, seq_len=s)['kv_cache_gb'] 
                for s in seq_lengths]
    ax2.plot(seq_lengths, kv_sizes, 'o-', label=f'KV Cache ({prec})', color=color, linewidth=2)

ax2.axhline(y=24 - estimate_model_memory(7.0)['weights_gb'], 
            color='orange', linestyle='--', alpha=0.7, label='Available VRAM (RTX 4090)')
ax2.set_xlabel('Sequence Length (tokens)')
ax2.set_ylabel('KV Cache Size (GB)')
ax2.set_title('KV Cache Scaling with Sequence Length\n(7B model)', fontweight='bold')
ax2.legend()
ax2.set_xscale('log', base=2)
ax2.set_xticks(seq_lengths)
ax2.set_xticklabels([str(s) for s in seq_lengths], rotation=45)

plt.tight_layout()
plt.savefig('memory_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Throughput Benchmark

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

if not TEST_MODE:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    
    print("Loading distilgpt2 for throughput benchmark...")
    tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained("distilgpt2")
    model.eval()
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"Using device: {device}")
    
    prompt = "Patient glucose reading:"
    batch_sizes = [1, 2, 4, 8]
    new_tokens = 20
    
    throughputs = []
    latencies = []
    
    for bs in batch_sizes:
        inputs = tokenizer([prompt] * bs, return_tensors="pt", padding=True).to(device)
        
        # Warmup
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=5, do_sample=False)
        
        # Benchmark
        times = []
        for _ in range(3):
            start = time.perf_counter()
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=new_tokens, do_sample=False)
            elapsed = time.perf_counter() - start
            times.append(elapsed)
        
        avg_time = np.mean(times)
        total_tokens = bs * new_tokens
        throughput = total_tokens / avg_time
        latency = avg_time * 1000 / bs  # ms per request
        
        throughputs.append(throughput)
        latencies.append(latency)
        print(f"Batch {bs:2d}: {throughput:6.1f} tok/s, {latency:6.1f} ms/req")

else:
    print("TEST_MODE: Using simulated throughput data")
    batch_sizes = [1, 2, 4, 8, 16, 32]
    # Realistic throughput curve: increases then plateaus
    throughputs = [45, 82, 148, 245, 310, 330]
    latencies = [22, 24, 27, 33, 43, 71]
    
    print(f"{'Batch':>6} {'Throughput':>12} {'Latency':>12}")
    for bs, tp, lat in zip(batch_sizes, throughputs, latencies):
        print(f"{bs:>6} {tp:>10.1f} t/s {lat:>10.1f} ms")

# Plot latency vs throughput tradeoff
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Throughput vs batch size
ax1 = axes[0]
ax1.plot(batch_sizes, throughputs, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Batch Size')
ax1.set_ylabel('Throughput (tokens/sec)')
ax1.set_title('Throughput vs Batch Size', fontweight='bold')
ax1.grid(True, alpha=0.3)
for bs, tp in zip(batch_sizes, throughputs):
    ax1.annotate(f'{tp:.0f}', (bs, tp), textcoords="offset points", xytext=(0, 10), ha='center')

# Latency vs batch size
ax2 = axes[1]
ax2.plot(batch_sizes, latencies, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('Latency per Request (ms)')
ax2.set_title('Latency vs Batch Size', fontweight='bold')
ax2.grid(True, alpha=0.3)

# Pareto frontier: throughput vs latency
ax3 = axes[2]
scatter = ax3.scatter(latencies, throughputs, c=batch_sizes, cmap='viridis', s=100, zorder=5)
ax3.plot(latencies, throughputs, 'k--', alpha=0.3)
for bs, lat, tp in zip(batch_sizes, latencies, throughputs):
    ax3.annotate(f'bs={bs}', (lat, tp), textcoords="offset points", xytext=(5, 5), fontsize=9)
plt.colorbar(scatter, ax=ax3, label='Batch Size')
ax3.set_xlabel('Latency per Request (ms)')
ax3.set_ylabel('Throughput (tokens/sec)')
ax3.set_title('Latency-Throughput Tradeoff\n(Pareto Frontier)', fontweight='bold')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('throughput_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Key insight: Batch size 4-8 is often the sweet spot for GAS")
print("   (5 agents can be batched together for ~3x throughput improvement)")

### 3.3 Cost Estimator

In [ ]:
def estimate_inference_cost(
    tokens_per_day: int,
    model_size: str = '7B',
    deployment: str = 'cloud_api',
    n_agents: int = 1
) -> dict:
    """
    Estimate daily inference cost for LLM deployment.
    
    Args:
        tokens_per_day: Total tokens processed per day
        model_size: '7B', '13B', '70B', 'gpt-4', 'gpt-3.5'
        deployment: 'cloud_api', 'self_hosted_a100', 'self_hosted_h100'
        n_agents: Number of concurrent agents
    
    Returns:
        dict with cost breakdown
    """
    # Cloud API pricing (per 1M tokens, 2024 estimates)
    api_pricing = {
        'gpt-4': {'input': 10.0, 'output': 30.0},
        'gpt-3.5': {'input': 0.5, 'output': 1.5},
        '7B': {'input': 0.2, 'output': 0.2},   # e.g., Together AI
        '13B': {'input': 0.3, 'output': 0.3},
        '70B': {'input': 0.9, 'output': 0.9},
    }
    
    # Self-hosted GPU costs (per hour)
    gpu_hourly = {
        'self_hosted_a100': 3.0,   # A100 80GB on-demand
        'self_hosted_h100': 5.0,   # H100 on-demand
        'self_hosted_rtx4090': 0.8, # RTX 4090 spot
    }
    
    # Tokens per second for self-hosted (rough estimates)
    self_hosted_tps = {
        '7B': {'self_hosted_a100': 2000, 'self_hosted_h100': 4000, 'self_hosted_rtx4090': 800},
        '13B': {'self_hosted_a100': 1200, 'self_hosted_h100': 2500, 'self_hosted_rtx4090': 400},
        '70B': {'self_hosted_a100': 300, 'self_hosted_h100': 700, 'self_hosted_rtx4090': None},
    }
    
    result = {
        'tokens_per_day': tokens_per_day,
        'model_size': model_size,
        'deployment': deployment,
        'n_agents': n_agents,
    }
    
    if deployment == 'cloud_api':
        pricing = api_pricing.get(model_size, api_pricing['7B'])
        # Assume 50/50 input/output split
        input_tokens = tokens_per_day * 0.6
        output_tokens = tokens_per_day * 0.4
        daily_cost = (input_tokens / 1e6 * pricing['input'] + 
                      output_tokens / 1e6 * pricing['output']) * n_agents
        result['daily_cost_usd'] = round(daily_cost, 4)
        result['monthly_cost_usd'] = round(daily_cost * 30, 2)
        result['annual_cost_usd'] = round(daily_cost * 365, 2)
        result['cost_per_1k_tokens'] = round(daily_cost / (tokens_per_day / 1000), 6)
    
    elif deployment in gpu_hourly:
        hourly_rate = gpu_hourly[deployment]
        tps = self_hosted_tps.get(model_size, {}).get(deployment, 1000)
        if tps is None:
            result['error'] = f'{model_size} does not fit on this GPU'
            return result
        
        # Hours needed to process tokens_per_day
        hours_needed = tokens_per_day / (tps * 3600)
        # But we need the GPU running 24/7 for real-time monitoring
        daily_cost = hourly_rate * 24 * n_agents  # always-on
        result['daily_cost_usd'] = round(daily_cost, 2)
        result['monthly_cost_usd'] = round(daily_cost * 30, 2)
        result['annual_cost_usd'] = round(daily_cost * 365, 2)
        result['gpu_utilization_pct'] = round(hours_needed / 24 * 100, 1)
        result['tokens_per_sec'] = tps
    
    return result


# GAS cost analysis
print("GAS Inference Cost Analysis")
print("=" * 60)
print("Assumptions: 5 agents, 12 CGM readings/hour, 200 tokens/reading")
print(f"= {5 * 12 * 24 * 200:,} tokens/day total")
print()

gas_tokens_per_day = 5 * 12 * 24 * 200  # 5 agents × 12 reads/hr × 24hr × 200 tokens

scenarios = [
    ('GPT-4 API', 'gpt-4', 'cloud_api'),
    ('GPT-3.5 API', 'gpt-3.5', 'cloud_api'),
    ('7B API (Together)', '7B', 'cloud_api'),
    ('7B Self-hosted A100', '7B', 'self_hosted_a100'),
    ('7B Self-hosted H100', '7B', 'self_hosted_h100'),
]

print(f"{'Scenario':<30} {'Daily':>10} {'Monthly':>12} {'Annual':>12}")
print("-" * 66)
for name, model, deploy in scenarios:
    r = estimate_inference_cost(gas_tokens_per_day, model, deploy, n_agents=5)
    if 'error' not in r:
        print(f"{name:<30} ${r['daily_cost_usd']:>8.2f} ${r['monthly_cost_usd']:>10.2f} ${r['annual_cost_usd']:>10.2f}")

In [ ]:
# Cost comparison visualization
import matplotlib.pyplot as plt
import numpy as np

scenarios_data = [
    ('GPT-4 API', 'gpt-4', 'cloud_api'),
    ('GPT-3.5 API', 'gpt-3.5', 'cloud_api'),
    ('7B API', '7B', 'cloud_api'),
    ('7B A100', '7B', 'self_hosted_a100'),
    ('7B H100', '7B', 'self_hosted_h100'),
]

names = []
monthly_costs = []
colors_list = []

for name, model, deploy in scenarios_data:
    r = estimate_inference_cost(gas_tokens_per_day, model, deploy, n_agents=5)
    if 'error' not in r:
        names.append(name)
        monthly_costs.append(r['monthly_cost_usd'])
        colors_list.append('#e74c3c' if 'API' in name else '#2ecc71')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, monthly_costs, color=colors_list, alpha=0.8, edgecolor='black')
ax.set_ylabel('Monthly Cost (USD)')
ax.set_title('GAS Monthly Inference Cost by Deployment Strategy\n(5 agents, 24/7 monitoring)', 
             fontweight='bold')

for bar, cost in zip(bars, monthly_costs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(monthly_costs)*0.01,
            f'${cost:.0f}', ha='center', fontweight='bold')

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Cloud API'),
                   Patch(facecolor='#2ecc71', label='Self-hosted')]
ax.legend(handles=legend_elements)
ax.set_yscale('log')
ax.set_ylabel('Monthly Cost (USD, log scale)')

plt.tight_layout()
plt.savefig('cost_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Self-hosted 7B on A100 is ~10-100x cheaper than GPT-4 API for GAS")

---
## 🏥 4. GAS Connection

### GAS Deployment Architecture

Running 5 GAS agents simultaneously requires careful resource planning. Let's calculate the full memory footprint:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# GAS Deployment Memory Calculator
def calculate_gas_deployment(
    model_size_b: float = 7.0,
    precision: str = 'fp16',
    n_agents: int = 5,
    seq_len: int = 4096,
    share_weights: bool = True
) -> dict:
    """
    Calculate memory for GAS multi-agent deployment.
    
    Args:
        share_weights: If True, all agents share one model copy (vLLM style)
    """
    base = estimate_model_memory(model_size_b, precision=precision, seq_len=seq_len, batch_size=1)
    
    if share_weights:
        # One model copy, but KV cache per agent
        total_weights = base['weights_gb']
        total_kv = base['kv_cache_gb'] * n_agents
        total_activations = base['activations_gb'] * n_agents
    else:
        # Separate model per agent (naive approach)
        total_weights = base['weights_gb'] * n_agents
        total_kv = base['kv_cache_gb'] * n_agents
        total_activations = base['activations_gb'] * n_agents
    
    total = total_weights + total_kv + total_activations
    
    return {
        'model_size': f'{model_size_b}B',
        'n_agents': n_agents,
        'share_weights': share_weights,
        'weights_gb': round(total_weights, 2),
        'kv_cache_gb': round(total_kv, 2),
        'activations_gb': round(total_activations, 3),
        'total_gb': round(total, 2),
        'fits_on_a100_80gb': total <= 72,  # 80GB with 10% headroom
        'fits_on_h100_80gb': total <= 72,
        'n_a100s_needed': int(np.ceil(total / 72)),
    }

print("GAS Deployment Memory Analysis")
print("=" * 60)

configs = [
    (7.0, 'fp16', False, 'Naive (separate models)'),
    (7.0, 'fp16', True, 'Shared weights (vLLM)'),
    (7.0, 'int8', True, 'Shared + int8 quant'),
    (7.0, 'int4', True, 'Shared + int4 quant'),
    (13.0, 'fp16', True, '13B shared weights'),
]

print(f"{'Config':<35} {'Weights':>8} {'KV':>8} {'Total':>8} {'A100s':>6}")
print("-" * 70)
for model_b, prec, share, name in configs:
    r = calculate_gas_deployment(model_b, prec, n_agents=5, share_weights=share)
    fits = '✅' if r['fits_on_a100_80gb'] else '❌'
    print(f"{name:<35} {r['weights_gb']:>6.1f}GB {r['kv_cache_gb']:>6.1f}GB {r['total_gb']:>6.1f}GB {r['n_a100s_needed']:>4} {fits}")

print("\n🏆 Recommended: 7B model, fp16, shared weights (vLLM) = fits on single A100!")

### GAS Infrastructure Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    GAS INFERENCE STACK                      │
│                                                             │
│  ┌─────────────┐  ┌─────────────┐  ┌─────────────────────┐  │
│  │ Monitor     │  │ Advisor     │  │ Safety Critic       │  │
│  │ Agent       │  │ Agent       │  │ Agent               │  │
│  └──────┬──────┘  └──────┬──────┘  └──────────┬──────────┘  │
│         │                │                     │            │
│  ┌──────┴────────────────┴─────────────────────┴──────────┐ │
│  │              vLLM Inference Server                      ││
│  │         (Continuous Batching + PagedAttention)          ││
│  │              Shared 7B Model Weights                    ││
│  └─────────────────────────────────────────────────────────┘│
│                           │                                 │
│  ┌────────────────────────┴────────────────────────────────┐│
│  │                  A100 80GB GPU                          ││
│  │  Weights: 14GB  │  KV Cache: 5×1GB  │  Activations: 1GB│ │
│  │  Total: ~20GB   │  Available: 60GB headroom             ││
│  └─────────────────────────────────────────────────────────┘│
└─────────────────────────────────────────────────────────────┘
```

**Key design decisions:**
1. **Shared weights**: All 5 agents use the same 7B model — saves 4× model memory
2. **Separate KV caches**: Each agent maintains its own conversation context
3. **Continuous batching**: When multiple agents need inference simultaneously, batch them
4. **int8 quantization**: Optional — halves memory with minimal quality loss

---
## ✅ 5. Checkpoint

### What We Built

| Component | Description | Reusable? |
|-----------|-------------|----------|
| `estimate_model_memory()` | GPU memory calculator for any LLM | ✅ Yes |
| Throughput benchmarker | tokens/sec vs batch size measurement | ✅ Yes |
| `estimate_inference_cost()` | Daily/monthly cost estimator | ✅ Yes |
| `calculate_gas_deployment()` | GAS-specific memory planner | ✅ Yes |

### Key Takeaways

1. **7B model in fp16 = ~14GB weights** — fits on a single A100 with room for 5 KV caches
2. **Shared weights + separate KV caches** is the right architecture for multi-agent systems
3. **Batch size 4-8** is the throughput sweet spot for GAS's 5 concurrent agents
4. **Self-hosted 7B is 10-100x cheaper** than GPT-4 API for always-on monitoring

### What Comes Next

**Chapter 3: Introduction to RL** — Now that we understand the hardware, we'll learn how to *train* agents to make better decisions using Reinforcement Learning. We'll build a glucose management environment and train a Q-learning agent.

```
Chapter 1 ✅ → Chapter 2 ✅ → Chapter 3 (RL Foundations) → ...
```